# exp13b — Attention: Stego vs Open vs Control

**Question:** Are attention entropy and within-KL at paragraph-start positions elevated specifically for *hidden* letter constraints (stego), or equally for any letter constraint (control)?

- **stego** — hidden acrostic, no marker  
- **open** — free reasoning, no constraint  
- **control** — visible letter constraint, same format as stego  

**Key comparison:** stego vs control.  
If `stego ≈ control` → signal is structural (any constraint), not stego-specific.  
If `stego ≠ control` → signal is stego-specific.

**Method:** Teacher-forced forward pass, `output_attentions=True` only.  
**Data:** `results/exp11/exp11_pairs_with_control.json`  
**Runtime:** ~1.5 h on A100.

In [ ]:
import os, gc, re, json, shutil
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive, userdata
    from huggingface_hub import login
    login(token=userdata.get('HF_TOKEN'))
    drive.mount('/content/drive')
    if os.path.basename(os.getcwd()) != 'stego_CoT':
        if not os.path.exists('stego_CoT'):
            os.system('git clone -q https://github.com/annareshetnyak799-netizen/stego_CoT.git')
        os.chdir('stego_CoT')
    os.system('pip install -q transformers accelerate')
    DRIVE_DIR = '/content/drive/MyDrive/stego_cot_results/exp13b'
    os.makedirs(DRIVE_DIR, exist_ok=True)

    # Restore input data from Drive if missing
    INPUT_LOCAL = 'results/exp11/exp11_pairs_with_control.json'
    if not os.path.exists(INPUT_LOCAL):
        src = '/content/drive/MyDrive/stego_cot_results/exp11/exp11_pairs_with_control.json'
        os.makedirs('results/exp11', exist_ok=True)
        shutil.copy(src, INPUT_LOCAL)
        print('Restored input data from Drive.')
else:
    DRIVE_DIR = None

import torch
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f'GPU: {gpu.name}  ({gpu.total_memory/1e9:.0f} GB)')
    assert gpu.total_memory > 30e9, 'Use A100 (40GB+)'
else:
    raise RuntimeError('No GPU')

from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = 'meta-llama/Llama-3.1-8B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map='auto',
    attn_implementation='eager',  # required for output_attentions=True
)
model.eval()
DEVICE   = next(model.parameters()).device
N_LAYERS = model.config.num_hidden_layers
print(f'Model loaded. Layers: {N_LAYERS}')

In [ ]:
def find_paragraph_starts(full_ids, plen):
    """Return token positions (in full_ids) of the first token of each paragraph."""
    response_ids = full_ids[plen:]
    tok_strs = [tokenizer.decode([t], skip_special_tokens=True) for t in response_ids]

    # Build character-level map: for each token, its start char position
    buf = ''
    char_starts = []
    for s in tok_strs:
        char_starts.append(len(buf))
        buf += s

    # Find paragraph start char positions: beginning + first non-ws char after each \n\n+
    para_chars = []
    i = 0
    while i < len(buf):
        while i < len(buf) and buf[i] in ' \t\n':
            i += 1
        if i >= len(buf):
            break
        para_chars.append(i)
        nn = buf.find('\n\n', i)
        if nn == -1:
            break
        i = nn + 2

    # Map char positions → token indices
    result = []
    for cp in para_chars:
        for ti in range(len(char_starts)):
            end = char_starts[ti + 1] if ti + 1 < len(char_starts) else len(buf)
            if char_starts[ti] <= cp < end:
                result.append(plen + ti)
                break
    return result


def pad_to(v, length):
    out = np.zeros(length)
    out[:len(v)] = v
    return out


def attention_metrics(attn_tuple, sent_pos, other_pos):
    """
    attn_tuple: tuple of N_LAYERS tensors, each (1, n_heads, seq, seq)
    Returns: ent_by_layer, kl_by_layer (lists of N_LAYERS floats)
    """
    EPS = 1e-10
    ent_by_layer, kl_by_layer = [], []

    for layer_attn in attn_tuple:
        A = layer_attn[0].float().cpu().numpy()  # (n_heads, seq, seq)
        n_heads, seq_len, _ = A.shape
        ent_vals, kl_vals = [], []

        for h in range(n_heads):
            sent_dists, other_dists = [], []

            for pos in sent_pos:
                if pos >= seq_len:
                    continue
                p = A[h, pos, :pos + 1].copy()
                # Entropy
                pn = np.clip(p, EPS, None); pn /= pn.sum()
                ent_vals.append(float(-np.sum(pn * np.log2(pn))))
                sent_dists.append(p)

            for pos in other_pos:
                if pos >= seq_len:
                    continue
                other_dists.append(A[h, pos, :pos + 1].copy())

            # Within-KL: mean distribution at sent vs mean at other
            if sent_dists and other_dists:
                L = max(
                    max(len(p) for p in sent_dists),
                    max(len(q) for q in other_dists)
                )
                P = np.mean([pad_to(p, L) for p in sent_dists], axis=0)
                Q = np.mean([pad_to(q, L) for q in other_dists], axis=0)
                P = np.clip(P, EPS, None); P /= P.sum()
                Q = np.clip(Q, EPS, None); Q /= Q.sum()
                kl_vals.append(float(np.sum(P * np.log(P / Q))))

        ent_by_layer.append(float(np.mean(ent_vals)) if ent_vals else 0.0)
        kl_by_layer.append(float(np.mean(kl_vals)) if kl_vals else 0.0)

    return ent_by_layer, kl_by_layer


print('Helpers defined.')

In [ ]:
@torch.no_grad()
def run_forward(token_ids, sent_pos, other_pos):
    """Teacher-forced forward pass, attention only."""
    ids_t = torch.tensor([token_ids], dtype=torch.long).to(DEVICE)
    out = model(
        ids_t,
        output_attentions=True,
        output_hidden_states=False,
        use_cache=False,
    )
    ent, kl = attention_metrics(out.attentions, sent_pos, other_pos)
    del out, ids_t
    torch.cuda.empty_cache(); gc.collect()
    return ent, kl


print('run_forward defined.')

In [ ]:
INPUT_FILE  = 'results/exp11/exp11_pairs_with_control.json'
OUTPUT_DIR  = 'results/exp13b'
RAW_FILE    = f'{OUTPUT_DIR}/exp13b_raw.json'
CKPT_EVERY  = 50
os.makedirs(OUTPUT_DIR, exist_ok=True)

with open(INPUT_FILE) as f:
    all_pairs = json.load(f)

triplets = [
    p for p in all_pairs
    if p.get('fidelity') == 1.0
    and p.get('control_fidelity') == 1.0
    and not p.get('control_explicit', True)
    and p.get('control_ids') is not None
    and len([s for s in re.split(r'\n{2,}', p.get('stego_text','').strip()) if s.strip()]) == len(p['payload'])
    and len([s for s in re.split(r'\n{2,}', p.get('control_text','').strip()) if s.strip()]) == len(p['payload'])
]
print(f'Valid triplets: {len(triplets)} / {len(all_pairs)}')

if not os.path.exists(RAW_FILE) and DRIVE_DIR:
    drive_ckpt = os.path.join(DRIVE_DIR, 'exp13b_raw.json')
    if os.path.exists(drive_ckpt):
        shutil.copy(drive_ckpt, RAW_FILE)
        print('Restored checkpoint from Drive.')

if os.path.exists(RAW_FILE):
    with open(RAW_FILE) as f:
        raw = json.load(f)
    results = raw.get('results', [])
    if results and 'pair_id' not in results[0]:
        print('Incompatible checkpoint (old format) — starting fresh.')
        results = []
        done_ids = set()
    else:
        done_ids = {r['pair_id'] for r in results}
        print(f'Resumed: {len(results)} done.')
else:
    results = []
    done_ids = set()


def save_checkpoint():
    data = {'n_triplets': len(results), 'skipped': skipped, 'results': results}
    with open(RAW_FILE, 'w') as f:
        json.dump(data, f)
    if DRIVE_DIR:
        shutil.copy(RAW_FILE, os.path.join(DRIVE_DIR, 'exp13b_raw.json'))


skipped = 0
last_ckpt = len(results)

for pair in triplets:
    pid = pair.get('arc_id', '') + '_' + pair.get('payload', '')
    if pid in done_ids:
        continue

    try:
        payload = pair['payload'].upper()
        n_para  = len(payload)
        rec     = {'pair_id': pid, 'payload': payload}

        for cond, ids_key, plen_key in [
            ('s', 'stego_ids',   'stego_plen'),
            ('o', 'open_ids',    'open_plen'),
            ('c', 'control_ids', 'control_plen'),
        ]:
            full_ids = pair[ids_key]
            plen     = pair[plen_key]
            sent_pos = find_paragraph_starts(full_ids, plen)

            # stego and control must have exactly n_para paragraphs (constrained format).
            # open is free-form — use all paragraph starts it has (any count ≥ 1).
            if cond in ('s', 'c'):
                if len(sent_pos) != n_para:
                    raise ValueError(f'{cond}: expected {n_para} paragraphs, found {len(sent_pos)}')
            else:
                if len(sent_pos) == 0:
                    raise ValueError('o: no paragraph starts found')

            resp_len  = len(full_ids) - plen
            sent_set  = set(sent_pos)
            other_pos = [plen + i for i in range(0, resp_len, 5) if plen + i not in sent_set]

            ent, kl = run_forward(full_ids, sent_pos, other_pos)

            for l in range(N_LAYERS):
                key = f'L{l}'
                if key not in rec:
                    rec[key] = {}
                rec[key][f'{cond}_ent'] = round(ent[l], 6)
                rec[key][f'{cond}_kl']  = round(kl[l],  6)

        results.append(rec)
        done_ids.add(pid)
        print(f'[{len(results):4d}] {pid[:40]}  payload={payload}')

    except Exception as e:
        skipped += 1
        print(f'  SKIP {pid[:40]}: {e}')

    if len(results) >= last_ckpt + CKPT_EVERY:
        save_checkpoint()
        last_ckpt = len(results)
        print(f'  >>> checkpoint ({len(results)} done, {skipped} skipped)')

save_checkpoint()
print(f'\nDone. {len(results)} triplets, {skipped} skipped.')

In [ ]:
with open(RAW_FILE) as f:
    raw = json.load(f)
results = raw['results']
n = len(results)
N_LAYERS = 32  # Llama-3.1-8B-Instruct
print(f'Analysing {n} triplets')

# Per-layer means
s_ent = np.zeros(N_LAYERS); o_ent = np.zeros(N_LAYERS); c_ent = np.zeros(N_LAYERS)
s_kl  = np.zeros(N_LAYERS); o_kl  = np.zeros(N_LAYERS); c_kl  = np.zeros(N_LAYERS)

# Per-pair means (for t-tests)
s_ent_pp, o_ent_pp, c_ent_pp = [], [], []
s_kl_pp,  o_kl_pp,  c_kl_pp  = [], [], []

for rec in results:
    se, oe, ce, sk, ok, ck = [], [], [], [], [], []
    for l in range(N_LAYERS):
        key = f'L{l}'
        if key not in rec: continue
        s_ent[l] += rec[key]['s_ent']; o_ent[l] += rec[key]['o_ent']; c_ent[l] += rec[key]['c_ent']
        s_kl[l]  += rec[key]['s_kl'];  o_kl[l]  += rec[key]['o_kl'];  c_kl[l]  += rec[key]['c_kl']
        se.append(rec[key]['s_ent']); oe.append(rec[key]['o_ent']); ce.append(rec[key]['c_ent'])
        sk.append(rec[key]['s_kl']);  ok.append(rec[key]['o_kl']);  ck.append(rec[key]['c_kl'])
    s_ent_pp.append(np.mean(se)); o_ent_pp.append(np.mean(oe)); c_ent_pp.append(np.mean(ce))
    s_kl_pp.append(np.mean(sk));  o_kl_pp.append(np.mean(ok));  c_kl_pp.append(np.mean(ck))

s_ent /= n; o_ent /= n; c_ent /= n
s_kl  /= n; o_kl  /= n; c_kl  /= n

SA, OA, CA = np.array(s_ent_pp), np.array(o_ent_pp), np.array(c_ent_pp)
SK, OK, CK = np.array(s_kl_pp),  np.array(o_kl_pp),  np.array(c_kl_pp)

# Paired t-tests
def ttest(a, b):
    t, p = stats.ttest_rel(a, b)
    return round(float(a.mean() - b.mean()), 6), round(float(t), 3), round(float(p), 6)

e_so = ttest(SA, OA); e_sc = ttest(SA, CA); e_co = ttest(CA, OA)
k_so = ttest(SK, OK); k_sc = ttest(SK, CK); k_co = ttest(CK, OK)

print(f'n={n}\n')
print('ENTROPY (bits)')
print(f'  stego={SA.mean():.4f}  open={OA.mean():.4f}  control={CA.mean():.4f}')
print(f'  stego-open:     diff={e_so[0]:+.4f}  t={e_so[1]}  p={e_so[2]}')
print(f'  stego-control:  diff={e_sc[0]:+.4f}  t={e_sc[1]}  p={e_sc[2]}  <- KEY')
print(f'  control-open:   diff={e_co[0]:+.4f}  t={e_co[1]}  p={e_co[2]}')
print()
print('WITHIN-KL')
print(f'  stego={SK.mean():.4f}  open={OK.mean():.4f}  control={CK.mean():.4f}')
print(f'  stego-open:     diff={k_so[0]:+.4f}  t={k_so[1]}  p={k_so[2]}')
print(f'  stego-control:  diff={k_sc[0]:+.4f}  t={k_sc[1]}  p={k_sc[2]}  <- KEY')
print(f'  control-open:   diff={k_co[0]:+.4f}  t={k_co[1]}  p={k_co[2]}')

In [ ]:
layers = np.arange(N_LAYERS)

# --- Figure 1: Entropy by layer ---
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(layers, s_ent, color='firebrick',  lw=2, label='stego')
ax.plot(layers, o_ent, color='steelblue',  lw=2, label='open')
ax.plot(layers, c_ent, color='darkorange', lw=2, label='control')
ax.set_xlabel('Layer'); ax.set_ylabel('Attention entropy (bits)')
ax.set_title(f'exp13b: Attention entropy by layer\n(n={n}, Llama-3.1-8B-Instruct)')
ax.legend()
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/exp13b_entropy.png', dpi=150)
plt.show()

# --- Figure 2: Within-KL by layer ---
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(layers, s_kl, color='firebrick',  lw=2, label='stego')
ax.plot(layers, o_kl, color='steelblue',  lw=2, label='open')
ax.plot(layers, c_kl, color='darkorange', lw=2, label='control')
ax.set_xlabel('Layer'); ax.set_ylabel('Within-KL (nats)')
ax.set_title(f'exp13b: Within-KL by layer\n(n={n}, Llama-3.1-8B-Instruct)')
ax.legend()
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/exp13b_within_kl.png', dpi=150)
plt.show()

print('Figures saved.')

In [ ]:
summary = {
    'experiment': 'exp13b — attention entropy and within-KL',
    'n_triplets': n,
    'skipped': raw.get('skipped', 0),
    'entropy': {
        'stego_mean':   round(float(SA.mean()), 6),
        'open_mean':    round(float(OA.mean()), 6),
        'control_mean': round(float(CA.mean()), 6),
        'stego_vs_open':    {'diff': e_so[0], 't': e_so[1], 'p': e_so[2]},
        'stego_vs_control': {'diff': e_sc[0], 't': e_sc[1], 'p': e_sc[2]},
        'control_vs_open':  {'diff': e_co[0], 't': e_co[1], 'p': e_co[2]},
    },
    'within_kl': {
        'stego_mean':   round(float(SK.mean()), 6),
        'open_mean':    round(float(OK.mean()), 6),
        'control_mean': round(float(CK.mean()), 6),
        'stego_vs_open':    {'diff': k_so[0], 't': k_so[1], 'p': k_so[2]},
        'stego_vs_control': {'diff': k_sc[0], 't': k_sc[1], 'p': k_sc[2]},
        'control_vs_open':  {'diff': k_co[0], 't': k_co[1], 'p': k_co[2]},
    },
}

out_path = f'{OUTPUT_DIR}/exp13b_summary.json'
with open(out_path, 'w') as f:
    json.dump(summary, f, indent=2)
print('Saved:', out_path)
print(json.dumps(summary, indent=2))

In [ ]:
# Google Drive backup
if IN_COLAB and DRIVE_DIR:
    import pathlib
    for fp in pathlib.Path(OUTPUT_DIR).glob('exp13b*'):
        shutil.copy(fp, os.path.join(DRIVE_DIR, fp.name))
    nb = '/content/stego_CoT/notebooks/exp13b_attention.ipynb'
    if os.path.exists(nb):
        shutil.copy(nb, os.path.join(DRIVE_DIR, 'exp13b_attention.ipynb'))
    print(f'Backed up to {DRIVE_DIR}')